In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_pickle("/home/work/sem-agent/MIR-WM811K/Python/WM811K.pkl")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 811457 entries, 0 to 811456
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   dieSize         811457 non-null  float64
 1   failureType     811457 non-null  object 
 2   lotName         811457 non-null  object 
 3   trainTestLabel  811457 non-null  object 
 4   waferIndex      811457 non-null  float64
 5   waferMap        811457 non-null  object 
dtypes: float64(2), object(4)
memory usage: 37.1+ MB


In [4]:
df.head()

,dieSize,failureType,lotName,trainTestLabel,waferIndex,waferMap
0,1683.0,none,lot1,Training,1.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,1683.0,none,lot1,Training,2.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
2,1683.0,none,lot1,Training,3.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
3,1683.0,none,lot1,Training,4.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
4,1683.0,none,lot1,Training,5.0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


In [9]:
df.iloc[0][5]

/tmp/ipykernel_574576/3296793768.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.iloc[0][5]


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8)

In [ ]:
# label_ontology
import typing

_ALIGN = {
    "Center":    "Center",
    "Donut":     "Donut",
    "Edge-Loc":  "Edge-Loc",
    "Edge-Ring": "Edge-Ring",
    "Loc":       "Loc",
    "Near-full": "Near-Full",
    "Scratch":   "Scratch",
    "Random":    "Random",
    "none":      "Normal",
}

SCENARIO_CLASSES = frozenset({"Center", "Edge-Ring", "Scratch"})
NORMAL_CLASS = "Normal"
EXCLUDED_CLASSES = frozenset({"Donut", "Edge-Loc", "Loc", "Near-Full", "Random"})

def to_kg_entity(label: str) -> str:
    norm = label.strip().replace("_", "-").lower()
    for raw, kg in _ALIGN.items():
        if norm in {raw.replace("_", "-").lower(), kg.replace("_", "-").lower()}:
            return kg
    raise KeyError(f"미정의 라벨: {label!r} — mapping_table과 정렬 사전 점검 필요")

def scope_of(kg_label: typing.Optional[str]) -> str:
    """scenario | normal | excluded | unlabeled"""
    if kg_label is None:
        return "unlabeled"
    if kg_label in SCENARIO_CLASSES:
        return "scenario"
    if kg_label == NORMAL_CLASS:
        return "normal"
    return "excluded"

In [ ]:
# wm811k_loader.py
import numpy as np
import pandas as pd
# from preprocess.label_ontology import to_kg_entity, scope_of

def load_wm811k(pkl_path: str) -> pd.DataFrame:
    df = pd.read_pickle(pkl_path)          # 811,457행
    df["wafer_id"] = df["waferIndex"].astype("Int64").astype(str)  # int 캐스팅 (1~25)
    df["lot_id"] = df["lotName"].astype(str)
    df["source"] = "wm811k"

    def _lbl(v):
        if v is None:
            return ""
        a = np.asarray(v, dtype=object).ravel()
        if not a.size:
            return ""
        s = str(a[0])
        return "" if s in ("0", "0.0", "nan", "None") else s   # 결측 표기 → 미라벨
    df["raw_label"] = df["failureType"].map(_lbl)

    df["has_label"] = df["raw_label"] != ""                # ~172,950행
    df["is_background"] = ~df["has_label"]                 # 라벨 미확인 = 배경 물량
    
    df["kg_label"] = df["raw_label"].where(df["has_label"]).map(
        lambda v: to_kg_entity(v) if isinstance(v, str) and v else None)  # NaN(truthy) 방어
    df["is_normal"] = df["kg_label"] == "Normal"           # 정상 모수 기본
    df["scope"] = df["kg_label"].map(lambda v: scope_of(v if isinstance(v, str) else None))
    return df[["source", "lot_id", "wafer_id", "waferMap",
               "dieSize", "kg_label", "has_label", "is_background", "is_normal", "scope"]]

In [ ]:
# render.py
import numpy as np, io, base64
from PIL import Image

_PALETTE = {0: (30, 30, 40), 1: (70, 170, 90), 2: (220, 70, 70)}  # 배경/통과/불량
OUT_SIZE = (256, 256)

def render_wafer_png(die_map: np.ndarray) -> tuple[bytes, tuple[int, int]]:
    """0/1/2 die map -> PNG bytes (메타 데이터 관리를 위해 원본 해상도도 함께 반환)"""
    arr = np.asarray(die_map, dtype=np.uint8)
    orig_hw = arr.shape
    rgb = np.zeros((*arr.shape, 3), dtype=np.uint8)
    for v, c in _PALETTE.items():
        rgb[arr == v] = c
    img = Image.fromarray(rgb, "RGB").resize(OUT_SIZE, Image.NEAREST)  # 보간 금지
    buf = io.BytesIO(); img.save(buf, "PNG")
    return buf.getvalue(), (orig_hw[1], orig_hw[0])

def to_base64_png(die_map: np.ndarray) -> tuple[str, tuple[int, int]]:
    png, orig = render_wafer_png(die_map)
    return base64.b64encode(png).decode(), orig